# ML-08 — Model Training and Hyperparameter Tuning

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshitttt077/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

**Author:** Harshit Kudhial (`@harshitttt077`) | **Track:** Machine Learning | **Lane:** Lane 2 Refresh Scoring


## 1. Model choices & architectures
We benchmark three models:
1. Regularized Logistic Regression (linear baseline)
2. Decision Tree Classifier (`max_depth=5`)
3. Random Forest Classifier (`n_estimators=200`, `max_depth=10`, `class_weight='balanced_subsample'`)


In [1]:
import json
with open('../outputs/model_results.json') as f:
    results = json.load(f)

for name, m in results['models'].items():
    print(f"Model: {name:20s} | Prec@50: {m['precision_at_50']:.4f} | Prec@20: {m['precision_at_20']:.4f} | ROC-AUC: {m['roc_auc']:.4f}")


Model: decision_tree        | Prec@50: 0.5400 | Prec@20: 0.5000 | ROC-AUC: 0.7415
Model: logistic_regression  | Prec@50: 0.4000 | Prec@20: 0.3500 | ROC-AUC: 0.7003
Model: random_forest        | Prec@50: 0.6800 | Prec@20: 0.7000 | ROC-AUC: 0.7474


## 2. Training code with client-holdout split
Training on 27,675 rows across 160 clients, validating on 2,325 rows across 40 holdout clients.


In [2]:
print(f"Train Rows: {results['train_rows']:,} | Test Rows: {results['test_rows']:,}")
print(f"Split Strategy: {results['split_strategy']}")
print(f"Best Model: {results['best_model']['name']} (Selection Metric: {results['best_model']['selection_metric']})")


Train Rows: 27,675 | Test Rows: 2,325
Split Strategy: client_holdout
Best Model: random_forest (Selection Metric: precision_at_50)


## 3. Results on holdout test set
Random Forest achieves 0.680 Precision@50 vs 0.240 Baseline (**2.83x lift**, +22 correct picks per month).


In [3]:
base_p50 = results['baseline']['baseline_precision_at_50']
rf_p50 = results['models']['random_forest']['precision_at_50']
print(f"Baseline Precision@50: {base_p50:.4f}")
print(f"Random Forest Prec@50: {rf_p50:.4f}")
print(f"Empirical Lift Ratio : {rf_p50/base_p50:.2f}x")


Baseline Precision@50: 0.2400
Random Forest Prec@50: 0.6800
Empirical Lift Ratio : 2.83x


## 4. Feature importance breakdown
Top features: `days_with_impressions` (16.0%), `log_impressions_90d` (12.8%), `avg_position` (10.8%), `content_age_days` (9.5%).


In [4]:
top_feats = results['best_model']['feature_importance_top'][:8]
for f in top_feats:
    print(f"  - {f['feature']:25s}: {f['importance']:.4f}")


  - days_with_impressions    : 0.1603
  - log_impressions_90d      : 0.1280
  - avg_position             : 0.1085
  - content_age_days         : 0.0948
  - word_count               : 0.0411
  - char_count               : 0.0402
  - log_clicks_90d           : 0.0329
  - ctr                      : 0.0324
